## Model Evaluation and Optimization

Indicino Employee Attrition Model Evaluation and Improvement

To improve the model's ability to identify employees at risk of attrition, several Random Forest models were developed and compared using Accuracy, Precision, Recall, and F1-score. Since the primary business objective is to identify employees likely to leave the organization, Recall was considered the most important evaluation metric, as it measures the proportion of actual attrition cases correctly identified by the model.

In [19]:
#Import necessary libraries
import pandas as pd
import numpy as np
import joblib

In [20]:
#Import cleaned dataset
df = pd.read_csv("../data/processed/cleaned_data.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,AgeBand
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,80,0,8,0,1,6,4,0,5,41-50
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,80,1,10,3,3,10,7,1,7,41-50
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,80,0,7,3,3,0,0,0,0,31-40
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,80,0,8,3,3,8,7,3,0,31-40
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,80,1,6,3,3,2,2,2,2,18-30


In [21]:
#Define features and target variable

df_model = pd.get_dummies(df, drop_first=True)

X = df_model.drop("Attrition", axis=1)
y = df_model["Attrition"]

#Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
#Baseline Model: Random Forest Classifier(1st iteration)

from sklearn.ensemble import RandomForestClassifier

rf_original = RandomForestClassifier(random_state=42)
rf_original.fit(X_train, y_train)


#Run predictions on the test set
y_pred_original = rf_original.predict(X_test)

In [23]:
#Model Improvement: Random Forest Classifier with Class Weighting(2nd iteration)

rf_balanced = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_balanced.fit(X_train, y_train)

#Run predictions on the test set
y_pred_balanced = rf_balanced.predict(X_test)

In [24]:
#Model Improvement: Random Forest Classifier with SMOTE Oversampling(3rd iteration)

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

rf_smote = RandomForestClassifier(random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)

#Run predictions on the test set
y_pred_smote = rf_smote.predict(X_test)

/Users/folashadeadekunle/Indicino-employee-attrition-prediction/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/folashadeadekunle/Indicino-employee-attrition-prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/folashadeadekunle/Indicino-employee-attrition-prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/folashadeadekunle/Indicino-employee-attrition-prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [25]:
#Model Improvement: Random Forest Classifier with SMOTE Oversampling and Class Weighting(4th iteration)

rf_smote_balanced = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_smote_balanced.fit(X_train_smote, y_train_smote)

#Run predictions on the test set
y_pred_smote_balanced = rf_smote_balanced.predict(X_test)

In [26]:
#Model Improvement: Random Forest Classifier with SMOTE Oversampling, Class Weighting, and Hyperparameter Tuning(5th iteration)

from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='recall', n_jobs=-1)
grid_search.fit(X_train_smote, y_train_smote)

rf_tuned = grid_search.best_estimator_
y_prod_tuned = rf_tuned.predict_proba(X_test)[:, 1]
y_pred_tuned = (y_prod_tuned >= 0.35).astype(int)


In [27]:
#Evaluate the performance of all models using Accuracy, Precision, Recall, F1-Score, classification report and confusion matrix

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Evaluate each model
models = {
    'Original RF': y_pred_original,
    'Balanced RF': y_pred_balanced,
    'SMOTE RF': y_pred_smote,
    'SMOTE Balanced RF': y_pred_smote_balanced,
    'Tuned RF': y_pred_tuned
}

results = []

for name, predictions in models.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-Score": f1_score(y_test, predictions, zero_division=0),
        "Confusion Matrix": confusion_matrix(y_test, predictions)
    })

results_df = pd.DataFrame(results)
results_df


,Model,Accuracy,Precision,Recall,F1-Score,Confusion Matrix
0,Original RF,0.870748,0.571429,0.102564,0.173913,"[[252, 3], [35, 4]]"
1,Balanced RF,0.877551,1.000000,0.076923,0.142857,"[[255, 0], [36, 3]]"
2,SMOTE RF,0.891156,0.684211,0.333333,0.448276,"[[249, 6], [26, 13]]"
3,SMOTE Balanced RF,0.891156,0.684211,0.333333,0.448276,"[[249, 6], [26, 13]]"
4,Tuned RF,0.778912,0.308824,0.538462,0.392523,"[[208, 47], [18, 21]]"


In [28]:
#Save the preferred model to disk
preferred_model = rf_tuned
joblib.dump(preferred_model, "../models/employee_attrition_prediction_model.pkl")


['../models/employee_attrition_prediction_model.pkl']

### Model Performance Summary

- Original Random Forest achieved an accuracy of 87.1%, but identified only 4 out of 39 employees who left the company (Recall = 10.3%). Although the model performed well in predicting employees who stayed, it was ineffective for attrition prediction

- Random Forest with Class Weighting alone did not improve the model's ability to detect attrition. While it achieved 100% precision, it correctly identified only 3 out of 39 employees who resigned (Recall = 7.7%), making it unsuitable for the business objective

- Random Forest with SMOTE helped to balance the training dataset significantly improved model performance. The model correctly identified 13 out of 39 employees who left (Recall = 33.3%) while maintaining good precision (68.4%) and the highest balanced performance (F1-score = 44.8%)

- For the Hyperparameter-Tuned Random Forest, after optimizing the model using GridSearchCV, the Recall increased further to 53.8%, allowing the model to correctly identify 21 out of 39 employees who resigned. This improvement came at the expense of lower precision (30.9%) and lower overall accuracy (77.9%) because the model classified more employees as being at risk, resulting in more false positives.

### Business Interpretation

The results demonstrate the trade-off between Precision and Recall, while the tuned model generated more false positives, it successfully identified a significantly higher number of employees likely to leave. From a business perspective, this trade-off may be acceptable because the cost of missing an employee who is about to resign is often greater than the cost of engaging an employee who ultimately decides to stay.

Although the SMOTE model achieved the highest balance between Precision and Recall (highest F1-score), the Tuned Random Forest model aligns more closely with the business objective of proactively identifying employees at risk of attrition. Therefore, it is selected as the preferred model for this project.